# BERT Based Research Paper Recommendation System

This notebook implements a semantic research paper recommendation system using:

- BERT embeddings
- Random Forest model
- Cosine similarity
- Semantic search techniques

The project aims to recommend similar research papers based on user queries using Natural Language Processing and Machine Learning.

In [8]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

# Load Dataset

In this step, we load the cleaned research paper dataset.

The dataset contains:
- paper titles
- cleaned abstracts/text
- category labels

In [10]:
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv("../Dataset/cleaned_data.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (5000, 14)


,id,title,category,category_code,published_date,updated_date,authors,first_author,summary,summary_word_count,combined_text,clean_text,char_count,word_count
0,abs-1302.3557v1,Approximations for Decision Making in the Demp...,Artificial Intelligence,cs.AI,2/13/13,2/13/13,['Mathias Bauer'],'Mathias Bauer',The computational complexity of reasoning with...,98,Approximations for Decision Making in the Demp...,approximation decision making dempstershafer t...,544,62
1,abs-1509.03247v1,An Epsilon Hierarchical Fuzzy Twin Support Vec...,Artificial Intelligence,cs.AI,9/10/15,9/10/15,['Arindam Chaudhuri'],'Arindam Chaudhuri',The research presents epsilon hierarchical fuz...,129,An Epsilon Hierarchical Fuzzy Twin Support Vec...,epsilon hierarchical fuzzy twin support vector...,881,104
2,abs-1603.02738v1,Learning to Blend Computer Game Levels,Artificial Intelligence,cs.AI,3/8/16,3/8/16,"['Matthew Guzdial', 'Mark Riedl']",'Matthew Guzdial',We present an approach to generate novel compu...,122,Learning to Blend Computer Game Levels We pres...,learning blend computer game level present app...,651,83
3,abs-1203.3493v1,Solving Hybrid Influence Diagrams with Determi...,Artificial Intelligence,cs.AI,3/15/12,3/15/12,"['Yijing Li', 'Prakash P. Shenoy']",'Yijing Li',We describe a framework and an algorithm for s...,93,Solving Hybrid Influence Diagrams with Determi...,solving hybrid influence diagram deterministic...,604,65
4,abs-1106.0667v1,Reasoning within Fuzzy Description Logics,Artificial Intelligence,cs.AI,6/3/11,6/3/11,['U. Straccia'],'U. Straccia',"Description Logics (DLs) are suitable, well-kn...",133,Reasoning within Fuzzy Description Logics Desc...,reasoning within fuzzy description logic descr...,682,83


# Check Dataset Columns

This helps verify all available columns before processing the dataset.

In [11]:
# ============================================================
# CHECK COLUMNS
# ============================================================

print(df.columns)

Index(['id', 'title', 'category', 'category_code', 'published_date',
       'updated_date', 'authors', 'first_author', 'summary',
       'summary_word_count', 'combined_text', 'clean_text', 'char_count',
       'word_count'],
      dtype='str')


# Handle Missing Values

Rows with missing:
- cleaned text
- category labels

are removed to improve model quality.

In [12]:
# ============================================================
# HANDLE MISSING VALUES
# ============================================================

df = df.dropna(subset=['clean_text'])

# Replace 'category' with your actual label column if needed
df = df.dropna(subset=['category'])

print("Remaining Shape:", df.shape)

Remaining Shape: (5000, 14)


# Use Smaller Dataset

A smaller dataset helps:
- faster execution
- reduced memory usage
- quicker experimentation

In [13]:
# ============================================================
# SAMPLE SMALLER DATASET
# ============================================================

df = df.sample(5000, random_state=42)

print("Using Dataset Shape:", df.shape)

Using Dataset Shape: (5000, 14)


# Load Pretrained BERT Model

We use:

`all-MiniLM-L6-v2`

This lightweight transformer model provides:
- fast embedding generation
- semantic understanding
- efficient NLP processing

In [14]:
# ============================================================
# LOAD BERT MODEL
# ============================================================

bert_model = SentenceTransformer('all-MiniLM-L6-v2')

print("BERT model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BERT model loaded successfully


# Generate BERT Embeddings

BERT converts text into dense numerical vectors called embeddings.

These embeddings capture:
- contextual meaning
- semantic relationships
- similarity between research papers

In [15]:
# ============================================================
# GENERATE BERT EMBEDDINGS
# ============================================================

texts = df['clean_text'].astype(str).tolist()

bert_embeddings = bert_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

print("Embedding Shape:", bert_embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embedding Shape: (5000, 384)


# Save Generated Embeddings

Embeddings are saved for future use to avoid recomputing them repeatedly.

In [16]:
# ============================================================
# SAVE EMBEDDINGS
# ============================================================

np.save("../model/bert_embeddings_5k.npy", bert_embeddings)

print("Embeddings saved successfully")

Embeddings saved successfully


# Encode Category Labels

Machine learning algorithms require numerical labels.

LabelEncoder converts category names into numeric values.

In [17]:
# ============================================================
# ENCODE LABELS
# ============================================================

encoder = LabelEncoder()

y = encoder.fit_transform(df['category'])

print("Number of Classes:", len(np.unique(y)))

Number of Classes: 1


# Split Dataset

The dataset is divided into:
- training data
- testing data

This helps prepare the model for learning.

In [18]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    bert_embeddings,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (4000, 384)
Testing Shape: (1000, 384)


# Train Random Forest Model

Random Forest is an ensemble machine learning algorithm that combines multiple decision trees for better predictions.

Advantages:
- robust performance
- handles high-dimensional data
- reduces overfitting
- efficient classification

In [19]:
# ============================================================
# TRAIN RANDOM FOREST MODEL
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully")

Random Forest model trained successfully


# Build Recommendation Function

This function:
1. Converts user query into BERT embedding
2. Computes cosine similarity
3. Retrieves the most similar research papers

In [20]:
# ============================================================
# RECOMMENDATION FUNCTION
# ============================================================

def recommend_papers(query, top_n=5):

    query_embedding = bert_model.encode([query])

    similarity_scores = cosine_similarity(
        query_embedding,
        bert_embeddings
    )

    top_indices = similarity_scores[0].argsort()[-top_n:][::-1]

    recommendations = df.iloc[top_indices][
        ['title', 'category']
    ]

    return recommendations

# Test Recommendation System

We test the semantic recommendation engine using a sample query.

In [21]:
# ============================================================
# TEST RECOMMENDATION SYSTEM
# ============================================================

query = "Deep Learning for Natural Language Processing"

results = recommend_papers(query)

print("Recommended Papers:\n")

print(results)

Recommended Papers:

                                                  title  \
899   Stress Test Evaluation of Transformer-based Mo...   
739   Autogenic Training With Natural Language Proce...   
4717  Neural-Symbolic Argumentation Mining: an Argum...   
2628  Towards Open-Text Semantic Parsing via Multi-T...   
4831                     Explainable Deep RDFS Reasoner   

                     category  
899   Artificial Intelligence  
739   Artificial Intelligence  
4717  Artificial Intelligence  
2628  Artificial Intelligence  
4831  Artificial Intelligence  


# Conclusion

In this notebook, we successfully implemented:

- BERT semantic embeddings
- Random Forest classification
- Semantic recommendation system
- Cosine similarity search

This project demonstrates the practical application of:
- Natural Language Processing
- Machine Learning
- Semantic Search
- Recommendation Systems